<a href="https://colab.research.google.com/github/saimamanzoor651/urdu-ocr-codesaviours-si26-saima/blob/main/SI26_Week4_Saima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q \
transformers==4.41.2 \
tokenizers==0.19.1 \
sentencepiece \
protobuf \
tiktoken \
torch \
torchvision \
pandas \
pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 126.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os
import torch
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset, DataLoader

from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    AdamW
)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [4]:
class UrduOCRDataset(Dataset):

    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        self.base_path = "/content/urdu-ocr-codesaviours-si26-saima"

        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        image_path = os.path.join(self.base_path, row["image"])

        image = Image.open(image_path).convert("RGB")

        pixel_values = self.processor(
            image,
            return_tensors="pt"
        ).pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=128,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze()

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [5]:
import os
print(os.listdir("/content"))

['.config', 'sample_data']


In [6]:
!git clone https://github.com/saimamanzoor651/urdu-ocr-codesaviours-si26-saima.git

Cloning into 'urdu-ocr-codesaviours-si26-saima'...
remote: Enumerating objects: 626, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 626 (delta 9), reused 0 (delta 0), pack-reused 604 (from 1)
Receiving objects: 100% (626/626), 4.63 MiB | 21.38 MiB/s, done.
Resolving deltas: 100% (81/81), done.


In [7]:
import os

print(os.listdir("/content"))

['.config', 'urdu-ocr-codesaviours-si26-saima', 'sample_data']


In [10]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed",
    use_fast=False
)

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Model loaded successfully!")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


In [8]:
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed",
    use_fast=False
)

dataset = UrduOCRDataset(
    "/content/urdu-ocr-codesaviours-si26-saima/labels.csv",
    processor
)

sample = dataset[0]

print("Sample pixel_values shape:", sample["pixel_values"].shape)
print("Sample labels shape:", sample["labels"].shape)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

print("Training samples:", train_size)
print("Testing samples:", test_size)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Dataset loaded: 203 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Training samples: 162
Testing samples: 41


In [11]:
from torch.utils.data import DataLoader
from transformers import AdamW

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4
)

optimizer = AdamW(
    model.parameters(),
    lr=5e-5
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))
print("Ready to train!")

/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Training batches: 41
Testing batches: 11
Ready to train!


In [12]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-"*30)

    for batch_idx, batch in enumerate(train_loader):

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(
                f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}"
            )

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

print("\nTraining Complete!")


Epoch 1/3
------------------------------
Batch 0/41 | Loss: 17.1879
Batch 10/41 | Loss: 0.1486
Batch 20/41 | Loss: 0.0004
Batch 30/41 | Loss: 0.0158
Batch 40/41 | Loss: 0.0019
Epoch 1 Average Loss: 0.5693

Epoch 2/3
------------------------------
Batch 0/41 | Loss: 0.0038
Batch 10/41 | Loss: 0.0002
Batch 20/41 | Loss: 0.0001
Batch 30/41 | Loss: 0.0000
Batch 40/41 | Loss: 0.0000
Epoch 2 Average Loss: 0.0006

Epoch 3/3
------------------------------
Batch 0/41 | Loss: 0.0000
Batch 10/41 | Loss: 0.0000
Batch 20/41 | Loss: 0.0000
Batch 30/41 | Loss: 0.0000
Batch 40/41 | Loss: 0.0000
Epoch 3 Average Loss: 0.0000

Training Complete!


In [13]:
model.eval()

print("=== Model Evaluation ===")
print()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        generated_ids = model.generate(pixel_values)

        generated_text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        actual_text = processor.batch_decode(
            labels,
            skip_special_tokens=True
        )

        for pred, actual in zip(generated_text, actual_text):

            total += 1

            if pred.strip() == actual.strip():
                correct += 1

            print("Predicted:", pred)
            print("Actual   :", actual)
            print()

accuracy = (correct / total) * 100 if total > 0 else 0

print(f"Accuracy: {accuracy:.1f}% ({correct}/{total})")

=== Model Evaluation ===



/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1168: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted: TODO
Actual   : TODO

Predicted:

In [14]:
from google.colab import drive

drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/SI26-urdu-ocr-model"

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print("Model saved successfully!")
print(save_path)

Mounted at /content/drive
Model saved successfully!
/content/drive/MyDrive/SI26-urdu-ocr-model
